In [1]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
import gc
import json

import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.utils import class_weight
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


from lightgbm import LGBMClassifier

from utils import reduce_mem_usage

In [3]:
with open(r'prep_data/unique_dict.txt', 'r', encoding='utf-8') as f:
    dict_key = json.load(f)
columns_for_enc = [dict_key[col] for col in dict_key.keys()]

In [4]:
feature_names = [f'{key}_{value}' for key, values in dict_key.items() for value in values[1:]]

In [5]:
cat_cols = list(dict_key.keys())

In [6]:
X = pd.read_parquet(r'raw_data/train_data_0.pq')
y = pd.read_csv(r'raw_data/target.csv')

In [7]:
X.shape, y.shape

((1974724, 61), (3000000, 2))

In [8]:
df = X.merge(y, how='inner', on='id')
df = df[:200000]
df.shape

(200000, 62)

In [9]:
y = df['flag']
X = df.drop(['flag'], axis=1)
X.shape, y.shape

((200000, 61), (200000,))

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((160000, 61), (40000, 61), (160000,), (40000,))

In [11]:
X_train_id = X_train[['id']].copy()

In [12]:
min_id, max_id = X_train['id'].min(), X_train['id'].max()
min_id, max_id

(0, 25880)

In [13]:
y_prep = df[(df['id'] >= min_id) & (df['id'] <= max_id)][['id', 'flag']].drop_duplicates(ignore_index=True)

In [14]:
y_prep = y_prep[y_prep['id'].isin(X_train['id'].values)]
y_prep.shape

(25346, 2)

In [15]:
y_train_agg = y_prep.drop(['id'], axis=1)

In [16]:
class_weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(y_train_agg['flag']), y=y_train_agg['flag'])
print(f'Веса рассчитанные compute_weights: {class_weights}')

Веса рассчитанные compute_weights: [ 0.51486959 17.31284153]


In [17]:
weights = {
    0: class_weights[0],
    1: class_weights[1]
}
weights 

{0: 0.5148695864142357, 1: 17.312841530054644}

In [18]:
y_prep_test = y_prep[y_prep['id'].isin(X_train['id'].values)]
y_prep_test.shape

(25346, 2)

In [19]:
y_test_agg = y_prep_test.drop(['id'], axis=1)

In [20]:
class MemoryReduceTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.stats_ = None

    def fit(self, X, y):
        self.stats_ = reduce_mem_usage(X)
        return self

    def transform(self, X):
        X = self.stats_
        return X

In [21]:
class AggregationTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self, df_init: pd.DataFrame, feature_names: list, group_col: str='id', agg_func: str='sum'):
        self.df_init = df_init
        self.feature_names = feature_names
        self.group_col = group_col
        self.agg_func = agg_func

    def fit(self, X, y):

        return self

    def transform(self, X):
        df_temp = pd.DataFrame(X, columns=self.feature_names).astype('int8')
        df_temp = pd.concat([self.df_init, df_temp], axis=1)
        df_temp = df_temp.groupby(self.group_col, as_index=False).agg(self.agg_func)
        df_temp = df_temp.drop(columns=['id'])
        X = df_temp.copy()
        return X

In [22]:
categorical_transformer = Pipeline(steps=[
    ('red_mem', MemoryReduceTransformer()),
    ('onehot', OneHotEncoder(categories=columns_for_enc, sparse_output=False, handle_unknown='ignore', drop='first')),
    ('agg_trans', AggregationTransformer(df_init=X_train_id, feature_names=feature_names)),
    ('scaler', StandardScaler()),
])

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, cat_cols),
    ]
)

In [24]:
clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LGBMClassifier(
        objective='binary',
        metric='auc',
        class_weight=weights ,
        num_leaves=56, 
        random_state=42
    ))
])

In [25]:
clf.fit(X_train, y_train_agg)

Используемая память до преобразования 0.073 Gb


  0%|          | 0/60 [00:00<?, ?it/s]

Используемая память после преобразования: 0.010 Gb
Количество используемой памяти уменьшено на 86.1%
[LightGBM] [Info] Number of positive: 732, number of negative: 24614
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013081 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2841
[LightGBM] [Info] Number of data points in the train set: 25346, number of used features: 319
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [26]:
y_pred_proba = clf.predict_proba(X_test)[:, 1]

In [27]:
auc_test = roc_auc_score(y_test_agg, y_pred_proba)
print(f'AUC на тесте: {auc_test:.4f}')

AUC на тесте: 0.9771
